# Publisher comparison for CERC queries
To better understand where the large difference in publicaton lists between SciVal and OpenAlex for the three CERC queries, we look in more detail at the publishers of the papers listed in each query. 

In [85]:
# import libraries 

import pandas as pd
import numpy as np
import json
import ast
import fuzzywuzzy
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
from datetime import datetime
from tqdm import tqdm 
from collections import defaultdict

# q1
q1_oa_raw = pd.read_csv('./cerc_results/query1.csv')
q1_sv_raw = pd.read_excel('./cerc_results/query1_SCIVAL.xlsx')

# q2
q2_oa_raw = pd.read_csv('./cerc_results/query2.csv')
q2_sv_raw = pd.read_excel('./cerc_results/query2_SCIVAL.xlsx')

# q3
q3_oa_raw = pd.read_csv('./cerc_results/query3.csv')
q3_sv_raw = pd.read_excel('./cerc_results/query3_SCIVAL.xlsx')

In [88]:
import os
print(os.path.abspath('./cerc_results/query1_SCIVAL.xlsx'))
print(os.getcwd())

c:\Users\tania\OneDrive - TM-IT Consulting Inc\Documents\juan\cerc_results\query1_SCIVAL.xlsx
c:\Users\tania\OneDrive - TM-IT Consulting Inc\Documents\juan


## Query 1

In [21]:
q1_oa_raw.columns

Index(['id', 'doi', 'title', 'display_name', 'relevance_score',
       'publication_year', 'publication_date', 'ids', 'language',
       'primary_location', 'type', 'indexed_in', 'open_access', 'authorships',
       'institutions', 'countries_distinct_count',
       'institutions_distinct_count', 'corresponding_author_ids',
       'corresponding_institution_ids', 'apc_list', 'apc_paid', 'fwci',
       'has_fulltext', 'cited_by_count', 'citation_normalized_percentile',
       'cited_by_percentile_year', 'biblio', 'is_retracted', 'is_paratext',
       'is_xpac', 'primary_topic', 'topics', 'keywords', 'concepts', 'mesh',
       'locations_count', 'locations', 'best_oa_location',
       'sustainable_development_goals', 'awards', 'funders', 'has_content',
       'content_urls', 'referenced_works_count', 'referenced_works',
       'related_works', 'abstract_inverted_index', 'counts_by_year',
       'updated_date', 'created_date'],
      dtype='object')

i want to keep id, doi, title, publication year, extract publisher name from primary location, extract is oa from open_access, fwci, cited_by_count only

In [ ]:
q1_oa_clean = q1_oa_raw
q1_oa_clean['publisher_TEMP'] = q1_oa_clean['primary_location'].apply(lambda x: ast.literal_eval(x))
q1_oa_clean['journal_name'] = q1_oa_clean['publisher_TEMP'].apply(lambda x: x['raw_source_name'])
q1_oa_clean['publisher_TEMP2'] = q1_oa_clean['publisher_TEMP'].apply(lambda x: x['source'])
q1_oa_clean['publisher_name'] = q1_oa_clean['publisher_TEMP2'].apply(lambda x: x['host_organization_name'] if x is not None else None)
q1_oa_clean['oa_TEMP'] = q1_oa_clean['open_access'].apply(lambda x: ast.literal_eval(x))
q1_oa_clean['oa_status'] = q1_oa_clean['oa_TEMP'].apply(lambda x: x['is_oa'])
q1_oa_clean = q1_oa_clean[['id', 'doi', 'title', 'publication_year', 'journal_name', 'publisher_name', 'oa_status', 'fwci', 'cited_by_count']]

#q1_oa_clean

,id,doi,title,publication_year,journal_name,publisher_name,oa_status,fwci,cited_by_count
0,https://openalex.org/W4327785494,https://doi.org/10.1109/tgrs.2023.3258666,SuperYOLO: Super Resolution Assisted Object De...,2023,IEEE Transactions on Geoscience and Remote Sen...,Institute of Electrical and Electronics Engineers,False,46.1470,284
1,https://openalex.org/W2998917588,https://doi.org/10.3390/rs12020266,Flood Detection and Susceptibility Mapping Usi...,2020,Remote Sensing,Multidisciplinary Digital Publishing Institute,True,21.4237,355
2,https://openalex.org/W4210348940,https://doi.org/10.3390/electronics11030431,Change Detection in Remote Sensing Image Data ...,2022,Electronics,Multidisciplinary Digital Publishing Institute,True,12.9916,123
3,https://openalex.org/W3041870784,https://doi.org/10.3390/ijerph17144933,Landslide Susceptibility Mapping Using Machine...,2020,International Journal of Environmental Researc...,Multidisciplinary Digital Publishing Institute,True,34.5674,145
4,https://openalex.org/W3147951545,https://doi.org/10.1109/jstars.2021.3070786,Cloud and Cloud Shadow Segmentation for Remote...,2021,IEEE Journal of Selected Topics in Applied Ear...,Institute of Electrical and Electronics Engineers,True,6.1917,65
...,...,...,...,...,...,...,...,...,...
963,https://openalex.org/W4285399363,https://doi.org/10.1149/ma2022-01391762mtgabs,Evaluation of Porous Media Gas Diffusion Model...,2022,ECS Meeting Abstracts,Institute of Physics,False,0.0000,0
964,https://openalex.org/W4381705073,https://doi.org/10.7554/elife.82538.sa0,Editor's evaluation: Phylodynamics of SARS-CoV...,2022,None,None,True,NaN,0
965,https://openalex.org/W4287921746,https://doi.org/10.3389/fcosc.2022.903132,Investing in monarch conservation: understandi...,2022,Frontiers in Conservation Science,Frontiers Media,True,0.0000,0
966,https://openalex.org/W4400364573,https://doi.org/10.1002/jia2.26305,Implementation research for today's HIV respon...,2024,Journal of the International AIDS Society,International AIDS Society,True,0.0000,0


In [72]:
q1_oa_publisher_groups = pd.DataFrame(q1_oa_clean.groupby(['publisher_name']).count()['doi']).sort_values(by='doi', ascending=False).reset_index().rename(columns={'doi': 'publication_count'})
q1_oa_publisher_groups

,publisher_name,publication_count
0,Elsevier BV,97
1,Institute of Electrical and Electronics Engineers,85
2,Springer Science+Business Media,49
3,Wiley,49
4,Multidisciplinary Digital Publishing Institute,32
...,...,...
70,Surveillance Studies Network,1
71,Universidade Federal de São Paulo,1
72,The Company of Biologists,1
73,University of Huddersfield Press,1


In [ ]:
q1_oa_journal_groups = pd.DataFrame(q1_oa_clean.groupby(['publisher_name', 'journal_name']).count()['doi']).sort_values(by='doi', ascending=False).reset_index().rename(columns={'doi': 'publication_count'})
pd.DataFrame(q1_oa_journal_groups)

,publisher_name,journal_name,publication_count
0,RELX Group (Netherlands),SSRN Electronic Journal,20
1,Cornell University,,17
2,Institute of Electrical and Electronics Engineers,IEEE Access,9
3,American Physical Society,Physical Review B,8
4,Multidisciplinary Digital Publishing Institute,Remote Sensing,8
...,...,...,...
387,Wiley,Naval Research Logistics (NRL),1
388,Wiley,North American Journal of Fisheries Management,1
389,Wiley,Ecological Applications,1
390,Wiley,Ecology Letters,1


In [86]:
q1_sv_raw

,id,doi,title,publication_year,publication_date,language,type,open_access,authorships,countries_distinct_count,...,fwci,has_fulltext,cited_by_count,biblio,primary_topic,topics,keywords,concepts,sustainable_development_goals,referenced_works_count
0,https://openalex.org/W2998917588,https://doi.org/10.3390/rs12020266,Flood Detection and Susceptibility Mapping Usi...,2020,2020-01-13,en,article,"{'is_oa': True, 'oa_status': 'gold', 'oa_url':...","[{'author_position': 'first', 'author': {'id':...",6.0,...,21.423700,0.0,355.0,"{'volume': '12', 'issue': '2', 'first_page': '...","{'id': 'https://openalex.org/T10930', 'display...","[{'id': 'https://openalex.org/T10930', 'displa...",[{'id': 'https://openalex.org/keywords/overfit...,"[{'id': 'https://openalex.org/C22019652', 'wik...","[{'id': 'https://metadata.un.org/sdg/13', 'dis...",129.0
1,https://openalex.org/W3035624836,https://doi.org/10.1109/cvpr42600.2020.01111,SAPIEN: A SimulAted Part-Based Interactive ENv...,2020,2020-06-01,en,preprint,"{'is_oa': False, 'oa_status': 'closed', 'oa_ur...","[{'author_position': 'first', 'author': {'id':...",2.0,...,20.258692,0.0,353.0,"{'volume': None, 'issue': None, 'first_page': ...","{'id': 'https://openalex.org/T11714', 'display...","[{'id': 'https://openalex.org/T11714', 'displa...",[{'id': 'https://openalex.org/keywords/robotic...,"[{'id': 'https://openalex.org/C34413123', 'wik...",[],82.0
2,https://openalex.org/W3108888189,https://doi.org/10.1175/jcli-d-19-1013.1,Changes in Annual Extremes of Daily Temperatur...,2020,2020-12-01,en,article,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...","[{'author_position': 'first', 'author': {'id':...",3.0,...,16.618400,1.0,329.0,"{'volume': '34', 'issue': '9', 'first_page': '...","{'id': 'https://openalex.org/T10029', 'display...","[{'id': 'https://openalex.org/T10029', 'displa...",[{'id': 'https://openalex.org/keywords/precipi...,"[{'id': 'https://openalex.org/C107054158', 'wi...","[{'display_name': 'Climate action', 'id': 'htt...",60.0
3,https://openalex.org/W3042819938,https://doi.org/10.1111/ele.13568,A processâ€based metacommunity framework link...,2020,2020-07-16,en,article,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...","[{'author_position': 'first', 'author': {'id':...",6.0,...,42.608000,0.0,323.0,"{'volume': '23', 'issue': '9', 'first_page': '...","{'id': 'https://openalex.org/T10005', 'display...","[{'id': 'https://openalex.org/T10005', 'displa...",[{'id': 'https://openalex.org/keywords/metacom...,"[{'id': 'https://openalex.org/C2778899818', 'w...","[{'id': 'https://metadata.un.org/sdg/15', 'dis...",86.0
4,https://openalex.org/W4327785494,https://doi.org/10.1109/tgrs.2023.3258666,SuperYOLO: Super Resolution Assisted Object De...,2023,2023-01-01,en,article,"{'is_oa': False, 'oa_status': 'closed', 'oa_ur...","[{'author_position': 'first', 'author': {'id':...",3.0,...,46.147000,0.0,284.0,"{'volume': '61', 'issue': None, 'first_page': ...","{'id': 'https://openalex.org/T11659', 'display...","[{'id': 'https://openalex.org/T11659', 'displa...",[{'id': 'https://openalex.org/keywords/compute...,"[{'id': 'https://openalex.org/C41008148', 'wik...","[{'id': 'https://metadata.un.org/sdg/10', 'dis...",62.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963,https://openalex.org/W3003673118,https://doi.org/10.1103/physrevc.101.025803,Mass measurements of neutron-rich gallium isot...,2020,2020-02-10,en,article,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...","[{'author_position': 'first', 'author': {'id':...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
964,https://openalex.org/W4225680939,https://doi.org/10.5194/egusphere-2022-99,"Antarctic sea ice over the past 130,000 years,...",2022,2022-04-06,en,review,"{'is_oa': True, 'oa_status': 'gold', 'oa_url':...","[{'author_position': 'first', 'author': {'id':...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
965,https://openalex.org/W4390919011,https://doi.org/10.1103/physrevresearch.6.l012008,Measurements o

i want to keep id, doi, title, publication year, extract publisher name from primary location, extract is oa from open_access, fwci, cited_by_count only

In [75]:
q1_sv_clean = q1_sv_raw

In [76]:
q1_sv_clean['id'][0]

'https://openalex.org/W2998917588'

## Query 2

## Query 3